# Import statements

In [83]:
# astropy
from astropy.table import Table, QTable, vstack, join
import astropy.units as u

# general
import numpy as np
import matplotlib.pyplot as plt

In [80]:
%load_ext autoreload
%autoreload 2

import co_to_h2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Fetching galaxy properties from PHANGS table

In [ ]:
# read in the PHANGS sample table -- note that public version with basic info doesn't have inclination, which we need
props_tab=Table.read('/Users/adignan/Documents/GitHub/co_to_h2/phangs_sample_table_v1p6.csv',header_start=184)

# pick our sample ahead of time
targets=['ngc0628', 'ngc1097', 'ngc3351', 'ngc3521', 'ngc3621', 'ngc3627',
         'ngc4254', 'ngc4321', 'ngc4536', 'ngc4569', 'ngc4579', 'ngc4631',
         'ngc4826', 'ngc5194', 'ngc5457', 'ngc7793']

props_filt=props_tab[np.isin(props_tab['name'], targets)]
props_filt

# For running the code with just one galaxy 

In [81]:
ngc0628_sfr=props_filt['props_sfr'][0]
ngc0628_mstar=props_filt['props_mstar'][0]
ngc0628_inc=props_filt['orient_incl'][0]
ngc0628_pa=props_filt['orient_posang'][0]
ngc0628_dist=props_filt['dist'][0]
ngc0628_reff=props_filt['size_reff'][0]

# Required files for running with just one galaxy
## These should all be uploaded to GitHub!

In [ ]:
# CO moment zero map
co='ngc0628_12m+7m+tp_co21_2.566192048arcsec_broad_mom0.fits'

# WISE 1, 3, and 4 maps 
w1_7p5='ngc0628_w1_gauss7p5.fits'
w1_15='ngc0628_w1_gauss15.fits'
w3_7p5='ngc0628_w3_gauss7p5.fits'
w4_15='ngc0628_w4_gauss15.fits'

# comparisons
massmap='/Users/adignan/Documents/GitHub/co_to_h2/NGC0628.ica2.massmap.sm_Msunpc2.fits'

# For one galaxy and one (or multiple) apertures

In [82]:
# calculate the molecular mass...
test_tab=co_to_h2.calc_m_mol(w1_7p5, w1_15, w3_7p5, w4_15, co,
               gal_sfr=ngc0628_sfr, gal_mstar=ngc0628_mstar, inc=ngc0628_inc,
               pa=ngc0628_pa,dist=ngc0628_dist,
               r_eff=ngc0628_reff)

# ...for each of the three methods...
methods=['gswlc','w3w1', 'w4w1']

for i, m in enumerate(methods):

    # ...then perform aperture photometry
    phot=co_to_h2.photometry(data=test_tab[f'm_mol_{m}'], ras_in=['01:36:41.7'], decs_in=['15:46:59'], radius= [7.7*u.arcsec], 
                             wcs_file=co, deg=False, method=m, reg_name=['ngc0628'])

    # this is to prevent seeing the same basic info columns three times
    if i == 0:
        phot_tab_final = phot.copy()
        phot_tab_final.keep_columns(['RA_J2000', 'Decl_J2000', 'radius', 'region name'])

    phot_tab_final[f'aperture sum ({m})']=phot[f'aperture sum ({m})']

phot_tab_final

/Users/adignan/Documents/GitHub/co_to_h2/co_to_h2.py:349: RuntimeWarning: invalid value encountered in log10
  q=np.log10(I_w3.value/I_w1.value)
/Users/adignan/miniconda3/lib/python3.12/site-packages/astropy/units/quantity.py:648: RuntimeWarning: invalid value encountered in log10
  result = super().__array_ufunc__(function, method, *arrays, **kwargs)


RA_J2000,Decl_J2000,radius,region name,aperture sum (gswlc),aperture sum (w3w1),aperture sum (w4w1)
,,arcsec,,,,
str10,str8,float64,str7[1],float64,float64,float64
01:36:41.7,15:46:59,7.7,ngc0628,16149.605000253277,16149.605000253277,16149.605000253277


# Running full code on multiple galaxies
## This requires input files that only I have (and will not upload to GitHub)

In [ ]:
# set up path to all WISE files
wise_path='/Users/adignan/research/phangs/wise_all/'

# read in regions
regions=Table.read('/Users/adignan/research/phangs/astr5470_input.csv')
regions_filt=regions[regions['sfrs_status']=='no'] # filter out ones with no ALMA data

# make main table for regions, parameters, and paths
merged_inner=join(left=regions_filt,right=props_filt, keys_left='galaxy',keys_right='name')

for row in merged_inner:

    # set up WISE input files
    w1_7p5_path=wise_path+row['galaxy']+'_w1_gauss7p5.fits'
    w1_15_path=wise_path+row['galaxy']+'_w1_gauss15.fits'
    w3_7p5_path=wise_path+row['galaxy']+'_w3_gauss7p5.fits'
    w4_15_path=wise_path+row['galaxy']+'_w4_gauss15.fits'

    # set up CO input file
    co_path=row['alma_mom0']

    # calculate total molecular mass maps
    maps_table=co_to_h2.calc_m_mol(w1_7p5_path, w1_15_path, w3_7p5_path, w4_15_path, co_path,
               gal_sfr=row['props_sfr'], gal_mstar=row['props_mstar'], inc=row['orient_incl'],
               pa=row['orient_posang'],dist=row['dist'], r_eff=row['size_reff'])
    
    # write results to a csv file
    maps_table.write(f'/Users/adignan/Documents/GitHub/co_to_h2/{row['galaxy']}_m_mol.csv', format='ascii.csv', overwrite=True)

<TableColumns names=('col0','galaxy','region','RA_J2000','Decl_J2000','radius','sfrs_status','RA_J2000_err','Decl_J2000_err','vla_path','dist_mpc','alma_cube','alma_mom0','name','pgc','alias','survey_astrosat_status','survey_astrosat_instrument','survey_astrosat_notes','survey_astrosat_references','survey_galex_status','survey_galex_instrument','survey_galex_notes','survey_galex_references','survey_halpha_status','survey_halpha_instrument','survey_halpha_notes','survey_halpha_references','survey_muse_status','survey_muse_instrument','survey_muse_notes','survey_muse_references','survey_kcwi_status','survey_kcwi_instrument','survey_kcwi_notes','survey_kcwi_references','survey_sitelle_status','survey_sitelle_instrument','survey_sitelle_notes','survey_sitelle_references','survey_typhoon_status','survey_typhoon_instrument','survey_typhoon_notes','survey_typhoon_references','survey_hst_status','survey_hst_instrument','survey_hst_notes','survey_hst_references','survey_irac_status','survey_ira

SystemExit: 

/Users/adignan/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


## Perform photometry on molecular mass maps and save to mega table

In [ ]:
methods=['gswlc','w3w1', 'w4w1']

phot_tabs=[]

for row in merged_inner:

    phot_tab_final = QTable()

    phot_tab_final['RA_J2000'] = [row['RA_J2000']]
    phot_tab_final['Decl_J2000'] = [row['Decl_J2000']]
    phot_tab_final['radius'] = [row['radius']]
    phot_tab_final['region name'] = [row['region']]

    for i, m in enumerate(methods):

        map=Table.read(f'/Users/adignan/research/phangs/m_mol/{row['galaxy']}_m_mol.csv')
        
        phot=co_to_h2.photometry(data=map[f'm_mol_{m}'], ras_in=row['RA_J2000'], decs_in=row['Decl_J2000'], radius= row['radius']*u.arcsec, wcs_file=row['alma_mom0'], 
                        deg=False, method=m, reg_name=row['region'])

        if i == 0:
            phot_tab = phot.copy()
            phot_tab.keep_columns(['RA_J2000', 'Decl_J2000', 'radius', 'region name'])

        phot_tab_final[f'aperture sum ({m})']=phot[f'aperture sum ({m})']
    
    phot_tabs.append(phot_tab_final)

phot_tab = vstack(phot_tabs)

phot_tab.write('/Users/adignan/Documents/GitHub/co_to_h2/m_mol_final.csv', format='csv', overwrite=True)

# Checking why M_mol is the same for all methods

In [75]:
ex_m_mol=Table.read('/Users/adignan/Documents/GitHub/co_to_h2/ngc0628_m_mol.csv')
size=int(np.sqrt(len(ex_m_mol)))
# plotmap(ex_m_mol['m_mol_gswlc'].reshape(size,size))
# plotmap(ex_m_mol['m_mol_w3w1'].reshape(size,size))
# plotmap(ex_m_mol['m_mol_w4w1'].reshape(size,size))
plt.figure()
plt.hist(ex_m_mol['m_mol_gswlc'],bins=1000,alpha=0.5)
plt.hist(ex_m_mol['m_mol_w3w1'],bins=1000,alpha=0.5)
plt.hist(ex_m_mol['m_mol_w4w1'],bins=1000,alpha=0.5)
plt.ylim(top=5000)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/adignan/Documents/GitHub/co_to_h2/ngc0628_m_mol.csv'